In [18]:
import random
from numpy import *
from functools import reduce

In [19]:
def sigmoid(inX):
    return 1.0/(1+exp(-inX))

# 神经元节点类
**Layer_index**代表节点所在的第几层

**Node_index**代表节点在该层中是第几个

In [20]:
class Node(object):
    def __init__(self, layer_index,node_index):
        self.layer_index = layer_index
        self.node_index = node_index
        self.downstream = []
        self.upstream = []
        self.output=0
        self.delta=0
    def set_output(self,output):
#         设置节点输出值 主要用于输入层，直接设置输入数据
        self.output=output
    def append_downstream_connection(self,conn):
#         添加一个下游连接 conn指向下一层节点
        self.downstream.append(conn)
    def append_upstream_connection(self,conn):
#         添加一个上游连接 conn指向上一层节点
        self.upstream.append(conn)
    def calc_output(self):
#         计算节点输出值 output=sigmoid(Σ(上游节点输出 * 连接权重)) conn在此处指代连接而不是节点
        output=reduce(lambda ret,conn:ret+conn.upstream_node.output * conn.weight,self.upstream,0)
        self.output=sigmoid(output)
    def calc_hidden_layer_delta(self):
#         计算隐藏层节点误差项 output * (1-output) * Σ(下游节点的误差项 * 连接权重) conn同样在此处指代连接而不是节点
        downstream_delta =reduce(
            lambda ret,conn:ret+conn.downstream_node.delta *conn.weight ,self.downstream,0
        )
        self.delta=self.output * (1-self.output) * downstream_delta
    def calc_output_layer_delta(self,label):
#          计算输出层节点误差项 output * (1- output) *( label - output)
        self.delta= self.output *(1-self.output) * (label - self.output)
    def __str__(self):
        node_str = '%u-%u: output: %f delta: %f' % (
        self.layer_index, self.node_index, self.output, self.delta)
        downstream_str = reduce(lambda ret, conn: ret + '\n\t' + str(conn),
                               self.downstream, '')
        upstream_str = reduce(lambda ret, conn: ret + '\n\t' + str(conn),
                             self.upstream, '')
        return node_str + '\n\tdownstream:' + downstream_str + '\n\tupstream:' + upstream_str


# 常数节点类

输出恒为1,用于实现偏置项

虽然常数节点没有输入(因此没有上游节点和上游连接)，但是因为有权重，反向传播需要更新权重，所以需要计算其delta用以更新权重，但是实际上由于常数节点输出恒为1，output*(1-output)恒为0，此处只是为了保持代码一致性，仍然保留计算。后续在神经网络的calc_delta中不用再进一步区分节点类型

In [21]:
class ConstNode(object):
    def __init__(self,layer_index,node_index):
        self.layer_index = layer_index
        self.node_index = node_index
        self.downstream = []
        # 偏置项无输入只有输出
        self.output=1
    def append_downstream_connection(self,conn):
        self.downstream.append(conn)
    def calc_hidden_layer_delta(self):
        downstream_delta =reduce(lambda ret,conn:ret+conn.downstream_node.delta *conn.weight,self.downstream,0.0)
        self.delta=self.output *(1-self.output) * downstream_delta
    def __str__(self):
        node_str = '%u-%u: output: 1' % (self.layer_index, self.node_index)
        downstream_str = reduce(lambda ret, conn: ret + '\n\t' + str(conn),
                               self.downstream, '')
        return node_str + '\n\tdownstream:' + downstream_str

# 神经网络层类

该类相当于一个集合，包括前面的两个子集：多个普通节点和一个常数节点（偏置）
注意此处除了创建之外，基本不需要处理常数节点，因为常数节点初始化时自动创建，并且输出值自动设置为1，计算输出值的时候可以忽略常数节点

In [22]:
class Layer(object):
    def __init__(self,layer_index,node_count):
        # 这里的node_index是不包含常数节点的节点数量
        self.layer_index = layer_index
        self.nodes=[]
        # 根据node_count创建指定数量的普通节点，循环从0到node_count-1,创建共node_count个普通节点
        for i in range(node_count):
            self.nodes.append(Node(layer_index,i))
        # 在第layer_index层的第node_count
        # 位置添加一个常数节点
        self.nodes.append(ConstNode(layer_index,node_count))
    def set_output(self,data):
        # 只用于输入层
        # data为输入数据列表
        for i in range(len(data)):
            self.nodes[i].set_output(data[i])
    def calc_output(self):
        for node in self.nodes[:-1]:
            node.calc_output()
    def dump(self):
#        打印该层所有节点信息
        for node in self.nodes:
            print(node)



# 连接类

代表神经网络两个节点之间的连接，每一条连接都有对应的权重,更新权重需要用到梯度

In [23]:
class Connection(object):
    def __init__(self,upstream_node,downstream_node):
        self.upstream_node=upstream_node
        self.downstream_node=downstream_node
        self.weight=random.uniform(-0.1,0.1)
        self.gradient=0.0
    def calc_gradient(self):
        self.gradient=self.upstream_node.output*self.downstream_node.delta
    def update_weight(self,rate):
        self.weight=self.weight+rate*self.gradient
    def get_gradient(self):
        return self.gradient
    def __str__(self):
         return '(%u-%u) -> (%u-%u) = %f' % (
            self.upstream_node.layer_index,
            self.upstream_node.node_index,
            self.downstream_node.layer_index,
            self.downstream_node.node_index,
            self.weight)


# 连接集合类

管理神经网络中所有连接

In [24]:
class Connections(object):
    def __init__(self):
        self.connections=[]
    def  add_connection(self,connection):
        self.connections.append(connection)
    def dump(self):
        for conn in self.connections:
            print(conn)

# 神经网络类

负责构建网络、训练和预测

In [25]:
class Network(object):
    # layers是一个列表,指定每一层的节点数,例如[2,4,2]表示2-4-2结构
    # 初始化流程：创建每一层-》创建层间连接
    def __init__(self,layers):
        self.connections=Connections()
        self.layers=[]
        layer_count=len(layers)
        node_count=0

        for i in range(layer_count):
            self.layers.append(Layer(i,layers[i]))

        # 遍历每一层，除了最后一层以外为当前层和下一层创建全连接,上一层含常数节点，下一层不含常数节点
        for layer in  range(layer_count-1):
            connections=[Connection(upstream_node,downstream_node)
                         for upstream_node in self.layers[layer].nodes
                         for downstream_node in self.layers[layer+1].nodes[:-1]
                         ]
            for conn in connections:
                self.connections.add_connection(conn)
        #         在上下游节点中注册
                conn.downstream_node.append_upstream_connection(conn)
                conn.upstream_node.append_downstream_connection(conn)
    def train(self,labels,data_set,rate,epoch):
        for i in range(epoch):
            for d in range(len(data_set)):
                self.train_one_sample(labels[d],data_set[d],rate)
    def train_one_sample(self,label,sample,rate):
        '''
        :param label: 样本标签
        :param sample: 样本特征
        :param rate: 学习率
        :return:训练步骤：1.前向传播预测 2.计算误差 3.反向传播更新权重
        '''
        self.predict(sample)
        self.calc_delta(label)
        self.update_weight(rate)
    #     输出层和隐藏层的误差需要分别计算
    def calc_delta(self,label):
        output_nodes=self.layers[-1].nodes
        for i in range(len(label)):
            output_nodes[i].calc_output_layer_delta(label[i])

        for layer in self.layers[-2::-1]:
            for node in layer.nodes:
                node.calc_hidden_layer_delta()
    def update_weight(self,rate):
        '''
        遍历除了最后一层的每一层（这些层都有下游连接，只需更新这些下游连接的权重依次过一遍即可）
        '''
        for layer in self.layers[:-1]:
            for node in layer.nodes:
                for conn in node.downstream:
                    conn.update_weight(rate)
    def calc_gradient(self):
        for layer in self.layers[:-1]:
            for node in layer.nodes:
                for conn in node.downstream:
                    conn.cal_gradient()
    def get_gradient(self,label,sample):
        self.predict(sample)
        self.calc_delta(label)
        self.calc_gradient()
    def predict(self,sample):
        self.layers[0].set_output(sample)
        for i in range(1,len(self.layers)):
            self.layers[i].calc_output()
        return list(map(lambda node: node.output, self.layers[-1].nodes[:-1]))
    def dump(self):
        for layer in self.layers:
            layer.dump()







#  数据规范化类

    用于将0-255的数字转换为8位二进制表示

    使用0.9表示1，0.1表示0，避免使用0和1的极端值

In [26]:
class Normalizer(object):
    def __init__(self):
        # 8位二进制的位掩码
        self.mask = [
            0x1, 0x2, 0x4, 0x8, 0x10, 0x20, 0x40, 0x80
        ]

    def norm(self, number):
        """
        将数字转换为二进制向量表示
        :param number: 0-255之间的整数
        :return: 长度为8的列表，每个元素是0.1或0.9
        """
        # 对每一位，如果该位为1则返回0.9，否则返回0.1
        return list(map(lambda m: 0.9 if number & m else 0.1, self.mask))

    def denorm(self, vec):
        """
        将二进制向量还原为数字
        :param vec: 长度为8的列表，每个元素是0-1之间的值
        :return: 还原后的整数
        """
        # 将连续值转换为二进制（>0.5视为1，否则为0）
        binary = list(map(lambda i: 1 if i > 0.5 else 0, vec))
        # 将二进制位乘以对应的掩码
        for i in range(len(self.mask)):
            binary[i] = binary[i] * self.mask[i]
        # 求和得到原始数字
        return reduce(lambda x, y: x + y, binary)

In [27]:
def mean_square_error(vec1, vec2):
    total = 0
    for i in range(len(vec1)):
        diff = vec1[i] - vec2[i]
        total += diff * diff
    return 0.5 * total

In [28]:
def gradient_check(network,sample_feature,sample_label):
    def network_error(vec1, vec2):
         total = 0
         for i in range(len(vec1)):
             diff = vec1[i] - vec2[i]
             total += diff * diff
         return 0.5 * total
    network.get_gradient(sample_feature,sample_label)
    for conn in network.connections.connections:
        actual_gradient=conn.get_gradient()
        epsilon = 0.0001
        conn.weight += epsilon
        error1 = network_error(network.predict(sample_feature), sample_label)


        conn.weight -= 2 * epsilon
        error2 = network_error(network.predict(sample_feature), sample_label)

        # 公式: (f(x+ε) - f(x-ε)) / (2ε)
        expected_gradient = (error2 - error1) / (2 * epsilon)

        # 打印对比结果
        print ('expected gradient: \t%f\nactual gradient: \t%f' % (
            expected_gradient, actual_gradient))




In [29]:

def train_data_set():
    """
    生成训练数据集
    注意：这里生成的是自编码器数据集，输入和输出相同
    """
    normalizer = Normalizer()
    data_set = []  # 存储输入数据
    labels = []    # 存储标签数据

    # 生成0-255之间的随机数，步长为8（只生成32个样本）
    for i in range(0, 256, 8):
        n = normalizer.norm(int(random.uniform(0, 256)))  # 随机生成数字并规范化
        data_set.append(n)   # 输入是规范化的向量
        labels.append(n)     # 标签也是相同的向量（自编码器学习恒等映射）

    return labels, data_set

In [30]:
def train(network):
    """
    训练网络的主函数
    :param network: 神经网络对象
    """
    # 生成训练数据
    labels, data_set = train_data_set()
    # 开始训练：学习率0.3，训练50轮
    network.train(labels, data_set, 0.3, 50)

In [31]:
def test(network, data):
    """
    测试单个数字的重构效果
    :param network: 神经网络对象
    :param data: 要测试的数字（0-255）
    """
    normalizer = Normalizer()
    norm_data = normalizer.norm(data)           # 规范化输入
    predict_data = network.predict(norm_data)   # 预测（重构）
    # 打印原始数字和预测重构的数字
    print ('\ttestdata(%u)\tpredict(%u)' % (
        data, normalizer.denorm(predict_data)))


In [32]:

def correct_ratio(network):
    """
    计算整个网络的正确率
    :param network: 神经网络对象
    """
    normalizer = Normalizer()
    correct = 0.0
    # 测试0-255所有数字
    for i in range(256):
        # 如果重构的数字等于原始数字，则计数
        if normalizer.denorm(network.predict(normalizer.norm(i))) == i:
            correct += 1.0
    # 打印正确率
    print ('correct_ratio: %.2f%%' % (correct / 256 * 100))


In [33]:
def gradient_check_test():
    """
    梯度检查测试函数
    创建一个2-2-2的小网络进行梯度检查
    """
    net = Network([2, 2, 2])
    sample_feature = [0.9, 0.1]  # 样本特征
    sample_label = [0.9, 0.1]    # 样本标签
    gradient_check(net, sample_feature, sample_label)

In [34]:
if __name__ == '__main__':
    net = Network([8, 3, 8])
    train(net)
    net.dump()
    correct_ratio(net)

0-0: output: 0.900000 delta: 0.000027
	downstream:
	(0-0) -> (1-0) = 0.018533
	(0-0) -> (1-1) = -0.073517
	(0-0) -> (1-2) = -0.092542
	upstream:
0-1: output: 0.900000 delta: 0.000002
	downstream:
	(0-1) -> (1-0) = 0.041057
	(0-1) -> (1-1) = -0.095635
	(0-1) -> (1-2) = 0.033515
	upstream:
0-2: output: 0.900000 delta: -0.000020
	downstream:
	(0-2) -> (1-0) = 0.041536
	(0-2) -> (1-1) = 0.071702
	(0-2) -> (1-2) = 0.060038
	upstream:
0-3: output: 0.900000 delta: -0.000021
	downstream:
	(0-3) -> (1-0) = 0.037915
	(0-3) -> (1-1) = 0.020586
	(0-3) -> (1-2) = 0.087533
	upstream:
0-4: output: 0.100000 delta: 0.000012
	downstream:
	(0-4) -> (1-0) = -0.068991
	(0-4) -> (1-1) = 0.057912
	(0-4) -> (1-2) = -0.081447
	upstream:
0-5: output: 0.100000 delta: -0.000021
	downstream:
	(0-5) -> (1-0) = 0.032180
	(0-5) -> (1-1) = 0.033998
	(0-5) -> (1-2) = 0.080076
	upstream:
0-6: output: 0.100000 delta: -0.000011
	downstream:
	(0-6) -> (1-0) = 0.012741
	(0-6) -> (1-1) = -0.024513
	(0-6) -> (1-2) = 0.063416
